In [3]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
Compliance Officer (Compliance Checker)
Module 3: Multi-Jurisdictional Compliance Checker

Data source: parsed_trades.json + M2_output.json

Update history:
1. UTI validation: first 20 characters must equal reporting party LEI
2. Special handling for T026-T028 status (CONDITIONAL / NOT_APPLICABLE)
3. Different field requirements for CFTC and EMIR
"""

import json
import re
from pathlib import Path
from datetime import datetime
from typing import Dict, List, Tuple, Any, Optional

# ============================================================
# LEI validation (ISO 7064 MOD 97-10)
# ============================================================

def lei_checksum(lei: str) -> bool:
    """
    Validate whether LEI conforms to ISO 7064 MOD 97-10 algorithm
    
    Steps:
    1. Convert the first 18 characters to numbers (A=10, B=11, ..., Z=35)
    2. Append "00" at the end to get the verification string
    3. Calculate 98 - (verification_number mod 97)
    4. The result must equal the last two digits of the LEI
    """
    if not lei or not isinstance(lei, str):
        return False
    
    lei = lei.strip().upper()
    
    # Basic format checks
    if len(lei) != 20:
        return False
    if not lei[:18].isalnum():
        return False
    if not lei[18:].isdigit():
        return False
    
    # Step 1: Convert first 18 characters to numbers
    converted = ""
    for char in lei[:18]:
        if char.isdigit():
            converted += char
        else:
            converted += str(ord(char) - ord('A') + 10)
    
    # Step 2: Append "00"
    verification_str = converted + "00"
    
    try:
        remainder = int(verification_str) % 97
        expected_checksum = 98 - remainder
        expected_checksum_str = f"{expected_checksum:02d}"  # Two digits, pad with 0
        actual_checksum = lei[18:20]
        return expected_checksum_str == actual_checksum
    except (ValueError, ZeroDivisionError):
        return False


def is_valid_lei(lei: Any, check_checksum: bool = True) -> Tuple[bool, str]:
    """Check whether LEI is valid"""
    if lei is None:
        return False, "LEI is missing (null)"
    if not isinstance(lei, str):
        return False, f"LEI must be string, got {type(lei).__name__}"
    lei = lei.strip()
    if lei == "":
        return False, "LEI is empty string"
    if lei == "MISSING_LEI":
        return False, "LEI is explicitly marked as MISSING_LEI"
    if len(lei) != 20:
        return False, f"LEI length is {len(lei)}, expected 20"
    
    # Basic format: first 18 alphanumeric, last 2 digits
    if not lei[:18].isalnum():
        return False, "LEI first 18 chars must be alphanumeric"
    if not lei[18:].isdigit():
        return False, "LEI last 2 chars must be digits"
    
    if check_checksum and not lei_checksum(lei):
        return False, "LEI checksum failed (ISO 7064 MOD 97-10)"
    
    return True, ""


def lei_format_only(lei: Any) -> Tuple[bool, str]:
    """Only check LEI format (do not verify checksum), used for UTI namespace validation"""
    return is_valid_lei(lei, check_checksum=False)


# ============================================================
# UTI validation (new rule)
# ============================================================

def validate_uti(uti: Optional[str], reporting_lei: Optional[str]) -> Tuple[bool, str]:
    """
    Validate UTI format
    
    Rules:
    1. Total length not exceeding 52 characters
    2. The first 20 characters must be a valid LEI format
    3. The first 20 characters must equal the reporting party LEI
    4. Suffix (after 21st character) may only contain uppercase letters, numbers, and hyphens
    """
    # Rule 1: UTI must not be None
    if uti is None:
        return False, "UTI is missing (null)"
    if not isinstance(uti, str):
        return False, f"UTI must be string, got {type(uti).__name__}"
    
    uti = uti.strip()
    if uti == "":
        return False, "UTI is empty string"
    
    # Rule 1: Total length not exceeding 52 characters
    if len(uti) > 52:
        return False, f"UTI length ({len(uti)}) exceeds 52 characters"
    
    if len(uti) < 20:
        return False, f"UTI length ({len(uti)}) is less than 20 characters"
    
    # Rule 2: The first 20 chars must be valid LEI format (format only, no checksum)
    namespace_lei = uti[:20]
    lei_valid, lei_error = lei_format_only(namespace_lei)
    if not lei_valid:
        return False, f"UTI namespace (first 20 chars) is not a valid LEI format: {lei_error}"
    
    # Rule 3: The first 20 chars must equal reporting party LEI
    if reporting_lei is None:
        return False, "Reporting counterparty LEI is missing, cannot validate UTI namespace"
    
    reporting_lei_clean = reporting_lei.strip().upper()
    if namespace_lei != reporting_lei_clean:
        return False, f"UTI namespace ({namespace_lei}) does not equal reporting LEI ({reporting_lei_clean})"
    
    # Rule 4: Suffix must only contain uppercase letters, numbers, and hyphen
    if len(uti) > 20:
        suffix = uti[20:]
        if not re.match(r'^[A-Z0-9\-]+$', suffix):
            return False, f"UTI suffix contains invalid characters: '{suffix}'"
    
    return True, ""


# ============================================================
# Timestamp validation
# ============================================================

def is_valid_timestamp(timestamp: Any) -> Tuple[bool, str]:
    """Check timestamp format (ISO 8601 UTC)"""
    if timestamp is None:
        return False, "Timestamp is missing (null)"
    if not isinstance(timestamp, str):
        return False, f"Timestamp must be string, got {type(timestamp).__name__}"
    timestamp = timestamp.strip()
    if timestamp == "":
        return False, "Timestamp is empty"
    try:
        if timestamp.endswith('Z'):
            timestamp_clean = timestamp[:-1] + '+00:00'
        else:
            timestamp_clean = timestamp
        datetime.fromisoformat(timestamp_clean)
        return True, ""
    except ValueError:
        return False, f"Invalid timestamp format (expected ISO 8601): '{timestamp}'"


def is_valid_date(date_str: Any) -> Tuple[bool, str]:
    """Check date format (YYYY-MM-DD)"""
    if date_str is None:
        return False, "Date is missing (null)"
    if not isinstance(date_str, str):
        return False, f"Date must be string, got {type(date_str).__name__}"
    date_str = date_str.strip()
    if date_str == "":
        return False, "Date is empty"
    if date_str == "9999-99-99":
        return True, ""  # Placeholder for perpetual contract
    try:
        datetime.strptime(date_str, "%Y-%m-%d")
        return True, ""
    except ValueError:
        return False, f"Invalid date format (expected YYYY-MM-DD): '{date_str}'"


# ============================================================
# Notional validation
# ============================================================

def is_valid_notional(amount: Any, currency: Any) -> Tuple[bool, str]:
    """Check notional amount"""
    if amount is None:
        return False, "Notional amount is missing"
    if not isinstance(amount, (int, float)):
        try:
            amount = float(amount)
        except (ValueError, TypeError):
            return False, f"Notional must be numeric, got {type(amount).__name__}"
    if amount <= 0:
        return False, f"Notional must be positive, got {amount}"
    if currency is not None and isinstance(currency, str):
        currency = currency.strip().upper()
        if currency == "INVALID_CCY":
            return False, "Invalid currency: INVALID_CCY"
    return True, ""


# ============================================================
# Helper functions: Trade type judgement
# ============================================================

def is_event_contract(trade: Dict) -> bool:
    """Determine if it is an event contract (T026-T028)"""
    asset_class = trade.get("asset_class", "")
    return asset_class == "EventContract"


def get_cftc_special_status(trade: Dict, trade_id: str) -> Optional[str]:
    """
    Get special CFTC status (for T026-T028)
    
    Returns:
    - "CONDITIONAL": Binary event contract traded on CFTC DCM
    - "NOT_APPLICABLE": Binary event contract not on CFTC DCM
    - None: Ordinary derivatives
    """
    if not is_event_contract(trade):
        return None
    
    platform = trade.get("platform", "").upper()
    platform_type = trade.get("platform_type", "").upper()
    
    # T027: Polymarket (Decentralized blockchain platform, accessed via VPN)
    if "POLYMARKET" in platform or "DECENTRALISED" in platform_type:
        return "NOT_APPLICABLE"
    
    # T026 & T028: Kalshi (CFTC-regulated DCM)
    return "CONDITIONAL"


def get_emir_special_status(trade: Dict) -> Optional[str]:
    """
    Get special EMIR status
    
    Returns:
    - "NOT_APPLICABLE": Binary event contracts classified as gambling, not subject to derivatives regulation
    - None: Ordinary derivatives
    """
    if is_event_contract(trade):
        return "NOT_APPLICABLE"
    return None


# ============================================================
# CFTC Compliance Check
# ============================================================

def check_cftc_compliance(trade: Dict, upi_result: Dict, trade_wrapper: Dict) -> Tuple[str, List[str], Optional[str]]:
    """
    CFTC mandatory compliance check
    
    Returns: (status, violations, special_note)
    status: COMPLIANT, NONCOMPLIANT, CONDITIONAL, NOT_APPLICABLE
    """
    trade_id = trade.get("trade_id")
    
    # Check for special status
    special_status = get_cftc_special_status(trade, trade_id)
    if special_status == "CONDITIONAL":
        note = "Binary event contract traded on CFTC-regulated DCM (Kalshi). Reporting obligations apply conditionally under Part 43/45. Refer to Module 4 for regulatory classification analysis."
        return "CONDITIONAL", [], note
    elif special_status == "NOT_APPLICABLE":
        return "NOT_APPLICABLE", [], "Binary event contract on offshore unregulated platform (Polymarket). No CFTC reporting obligation."
    
    violations = []
    
    # 1. UTI validation (new rule)
    uti_valid, uti_error = validate_uti(trade.get("uti"), trade.get("reporting_counterparty_lei"))
    if not uti_valid:
        violations.append(f"MISSING_OR_INVALID_UTI: {uti_error}")
    
    # 2. LEI - Reporting party
    lei_valid, lei_error = is_valid_lei(trade.get("reporting_counterparty_lei"))
    if not lei_valid:
        violations.append(f"INVALID_LEI_REPORTING: {lei_error}")
    
    # 3. LEI - Other counterparty
    other_lei = trade.get("other_counterparty_lei")
    if other_lei and other_lei != "MISSING_LEI":
        lei_other_valid, lei_other_error = is_valid_lei(other_lei)
        if not lei_other_valid:
            violations.append(f"INVALID_LEI_OTHER: {lei_other_error}")
    
    # 4. Execution timestamp
    ts_valid, ts_error = is_valid_timestamp(trade.get("execution_timestamp"))
    if not ts_valid:
        violations.append(f"INVALID_EXECUTION_TIMESTAMP: {ts_error}")
    
    # 5. Effective date
    effective_valid, effective_error = is_valid_date(trade.get("effective_date"))
    if not effective_valid:
        violations.append(f"INVALID_EFFECTIVE_DATE: {effective_error}")
    
    # 6. Maturity date
    maturity_valid, maturity_error = is_valid_date(trade.get("maturity_date"))
    if not maturity_valid:
        violations.append(f"INVALID_MATURITY_DATE: {maturity_error}")
    
    # 7. Notional
    notional_valid, notional_error = is_valid_notional(
        trade.get("notional_amount"),
        trade.get("notional_currency")
    )
    if not notional_valid:
        violations.append(f"INVALID_NOTIONAL: {notional_error}")
    
    # 8. Action type
    action_type = trade.get("action_type")
    valid_actions = ["NEW", "MODIFY", "CANCEL", "ERROR", "AMEND", "TERMINATE"]
    if action_type not in valid_actions:
        violations.append(f"INVALID_ACTION_TYPE: {action_type}")
    
    # 9. Cleared indicator
    cleared = trade.get("cleared")
    if cleared not in [True, False]:
        violations.append(f"INVALID_CLEARED_INDICATOR: {cleared}")
    
    # 10. UPI status
    upi_status = upi_result.get("status", "UNKNOWN")
    if upi_status == "NOT_FOUND":
        violations.append("UPI_NOT_FOUND")
    elif upi_status == "NO_PRODUCT_DEFINITION":
        violations.append("UPI_NO_PRODUCT_DEFINITION")
    elif upi_status not in ["FOUND", "NO_PRODUCT_DEFINITION"]:
        violations.append(f"UPI_UNKNOWN_STATUS: {upi_status}")
    # status == "FOUND" is normal, do not add violation
    
    status = "COMPLIANT" if len(violations) == 0 else "NONCOMPLIANT"
    return status, violations, None


# ============================================================
# EMIR Compliance Check
# ============================================================

def check_emir_compliance(trade: Dict, upi_result: Dict) -> Tuple[str, List[str], Optional[str]]:
    """
    EMIR recommended compliance check
    
    Returns: (status, violations, special_note)
    status: COMPLIANT, NONCOMPLIANT, NOT_APPLICABLE
    """
    # Check for special status
    special_status = get_emir_special_status(trade)
    if special_status == "NOT_APPLICABLE":
        return "NOT_APPLICABLE", [], "Binary event contracts classified as gambling under GlüStV 2021 framework. Not subject to EMIR trade reporting."
    
    violations = []
    
    # 1. UTI validation
    uti_valid, uti_error = validate_uti(trade.get("uti"), trade.get("reporting_counterparty_lei"))
    if not uti_valid:
        violations.append(f"MISSING_OR_INVALID_UTI: {uti_error}")
    
    # 2. LEI - Reporting party
    lei_valid, lei_error = is_valid_lei(trade.get("reporting_counterparty_lei"))
    if not lei_valid:
        violations.append(f"INVALID_LEI_REPORTING: {lei_error}")
    
    # 3. LEI - Other counterparty
    other_lei = trade.get("other_counterparty_lei")
    if other_lei and other_lei != "MISSING_LEI":
        lei_other_valid, lei_other_error = is_valid_lei(other_lei)
        if not lei_other_valid:
            violations.append(f"INVALID_LEI_OTHER: {lei_other_error}")
    
    # 4. Execution timestamp
    ts_valid, ts_error = is_valid_timestamp(trade.get("execution_timestamp"))
    if not ts_valid:
        violations.append(f"INVALID_EXECUTION_TIMESTAMP: {ts_error}")
    
    # 5. Effective date
    effective_valid, effective_error = is_valid_date(trade.get("effective_date"))
    if not effective_valid:
        violations.append(f"INVALID_EFFECTIVE_DATE: {effective_error}")
    
    # 6. Maturity date
    maturity_valid, maturity_error = is_valid_date(trade.get("maturity_date"))
    if not maturity_valid:
        violations.append(f"INVALID_MATURITY_DATE: {maturity_error}")
    
    # 7. Notional
    notional_valid, notional_error = is_valid_notional(
        trade.get("notional_amount"),
        trade.get("notional_currency")
    )
    if not notional_valid:
        violations.append(f"INVALID_NOTIONAL: {notional_error}")
    
    # 8. Action type
    action_type = trade.get("action_type")
    valid_actions = ["NEW", "MODIFY", "CANCEL", "ERROR", "AMEND", "TERMINATE"]
    if action_type not in valid_actions:
        violations.append(f"INVALID_ACTION_TYPE: {action_type}")
    
    # 9. Cleared indicator
    cleared = trade.get("cleared")
    if cleared not in [True, False]:
        violations.append(f"INVALID_CLEARED_INDICATOR: {cleared}")
    
    # 10. UPI status
    upi_status = upi_result.get("status", "UNKNOWN")
    if upi_status == "NOT_FOUND":
        violations.append("UPI_NOT_FOUND")
    elif upi_status == "NO_PRODUCT_DEFINITION":
        violations.append("UPI_NO_PRODUCT_DEFINITION")
    elif upi_status not in ["FOUND", "NO_PRODUCT_DEFINITION"]:
        violations.append(f"UPI_UNKNOWN_STATUS: {upi_status}")
    
    # 11. EMIR specific field: collateral portfolio code
    collateral = trade.get("collateral_portfolio_code")
    if not collateral:
        violations.append("MISSING_COLLATERAL_CODE")
    
    # 12. EMIR specific field: initial margin posted
    init_margin = trade.get("initial_margin_posted")
    if init_margin is None:
        violations.append("MISSING_INITIAL_MARGIN")
    elif not isinstance(init_margin, (int, float)):
        violations.append(f"INVALID_INITIAL_MARGIN: {init_margin}")
    
    # 13. EMIR specific field: variation margin posted
    var_margin = trade.get("variation_margin_posted")
    if var_margin is None:
        violations.append("MISSING_VARIATION_MARGIN")
    elif not isinstance(var_margin, (int, float)):
        violations.append(f"INVALID_VARIATION_MARGIN: {var_margin}")
    
    status = "COMPLIANT" if len(violations) == 0 else "NONCOMPLIANT"
    return status, violations, None


# ============================================================
# Main processing flow
# ============================================================

def main():
    print("=" * 60)
    print("Compliance Officer (Compliance Checker)")
    print("Module 3: Multi-Jurisdictional Compliance Checker")
    print("=" * 60)
    
    # File paths
    work_dir = Path("/Users/sunyifan/Desktop/6822 Task3")
    a_file = work_dir / "parsed_trades.json"
    b_file = work_dir / "M2_output.json"
    output_file = work_dir / "compliance_report.json"
    
    print(f"\n📂 Working directory: {work_dir}")
    print(f"   Data from Student A: {a_file}")
    print(f"   Data from Student B: {b_file}")
    
    # Check files
    if not a_file.exists():
        print(f"\n❌ Error: {a_file} not found")
        for f in work_dir.glob("*.json"):
            print(f"      - {f.name}")
        return
    if not b_file.exists():
        print(f"\n❌ Error: {b_file} not found")
        return
    
    # Load data
    print("\n📖 Loading data...")
    with open(a_file, 'r', encoding='utf-8') as f:
        a_data = json.load(f)
    print(f"   ✅ Loaded {len(a_data)} trades (Student A)")
    
    with open(b_file, 'r', encoding='utf-8') as f:
        b_data = json.load(f)
    print(f"   ✅ Loaded {len(b_data)} UPI results (Student B)")
    
    # Build UPI index
    upi_index = {item["trade_id"]: item for item in b_data}
    
    # Test LEI validation
    test_lei = "5493001KJTIIGC8Y1R12"
    print(f"\n   [Test] LEI Check: {test_lei}")
    print(f"   Result: {lei_checksum(test_lei)} (should be True)")
    
    # Perform compliance checks
    print("\n🔍 Performing compliance checks...")
    
    results = {
        "summary": {
            "total_trades": len(a_data),
            "cftc_compliant": 0,
            "cftc_noncompliant": 0,
            "cftc_conditional": 0,
            "cftc_not_applicable": 0,
            "emir_compliant": 0,
            "emir_noncompliant": 0,
            "emir_not_applicable": 0,
            "violation_type_counts": {}
        },
        "details": []
    }
    
    for trade_wrapper in a_data:
        trade = trade_wrapper.get("normalized_trade")
        if not trade:
            print(f"   ⚠️ Warning: Trade {trade_wrapper.get('trade_id')} missing normalized_trade")
            continue
        
        trade_id = trade.get("trade_id")
        upi_result = upi_index.get(trade_id, {"status": "MISSING"})
        
        # CFTC compliance
        cftc_status, cftc_violations, cftc_note = check_cftc_compliance(trade, upi_result, trade_wrapper)
        
        # EMIR compliance
        emir_status, emir_violations, emir_note = check_emir_compliance(trade, upi_result)
        
        # Statistics
        if cftc_status == "COMPLIANT":
            results["summary"]["cftc_compliant"] += 1
        elif cftc_status == "CONDITIONAL":
            results["summary"]["cftc_conditional"] += 1
        elif cftc_status == "NOT_APPLICABLE":
            results["summary"]["cftc_not_applicable"] += 1
        else:
            results["summary"]["cftc_noncompliant"] += 1
        
        if emir_status == "COMPLIANT":
            results["summary"]["emir_compliant"] += 1
        elif emir_status == "NOT_APPLICABLE":
            results["summary"]["emir_not_applicable"] += 1
        else:
            results["summary"]["emir_noncompliant"] += 1
        
        for v in cftc_violations + emir_violations:
            v_code = v.split(":")[0] if ":" in v else v
            results["summary"]["violation_type_counts"][v_code] = \
                results["summary"]["violation_type_counts"].get(v_code, 0) + 1
        
        results["details"].append({
            "trade_id": trade_id,
            "cftc": {
                "status": cftc_status,
                "violations": cftc_violations,
                "special_note": cftc_note
            },
            "emir": {
                "status": emir_status,
                "violations": emir_violations,
                "special_note": emir_note
            },
            "upi_info": {
                "status": upi_result.get("status"),
                "matched_template": upi_result.get("matched_template"),
                "upi_code": upi_result.get("upi_code")
            },
            "trade_summary": {
                "asset_class": trade.get("asset_class"),
                "instrument_type": trade.get("instrument_type"),
                "use_case": trade.get("use_case"),
                "platform": trade.get("platform"),
                "cleared": trade.get("cleared")
            }
        })
    
    # Print summary
    print("\n" + "=" * 60)
    print("📊 Compliance Check Summary")
    print("=" * 60)
    print(f"   Total trades: {results['summary']['total_trades']}")
    print()
    print(f"   CFTC status distribution:")
    print(f"      - COMPLIANT: {results['summary']['cftc_compliant']}")
    print(f"      - NONCOMPLIANT: {results['summary']['cftc_noncompliant']}")
    print(f"      - CONDITIONAL: {results['summary']['cftc_conditional']}")
    print(f"      - NOT_APPLICABLE: {results['summary']['cftc_not_applicable']}")
    print()
    print(f"   EMIR status distribution:")
    print(f"      - COMPLIANT: {results['summary']['emir_compliant']}")
    print(f"      - NONCOMPLIANT: {results['summary']['emir_noncompliant']}")
    print(f"      - NOT_APPLICABLE: {results['summary']['emir_not_applicable']}")
    
    if results['summary']['violation_type_counts']:
        print("\n   📋 Violation Type Statistics:")
        for v_code, count in sorted(results['summary']['violation_type_counts'].items(), key=lambda x: -x[1]):
            print(f"      - {v_code}: {count} times")
    
    # Save results
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(results, f, indent=2, ensure_ascii=False)
    
    print(f"\n💾 Detailed report saved to: {output_file}")
    print("\n✅ Compliance check complete! Please send compliance_report.json to Student D")


if __name__ == "__main__":
    main()

Compliance Officer (Compliance Checker)
Module 3: Multi-Jurisdictional Compliance Checker

📂 Working directory: /Users/sunyifan/Desktop/6822 Task3
   Data from Student A: /Users/sunyifan/Desktop/6822 Task3/parsed_trades.json
   Data from Student B: /Users/sunyifan/Desktop/6822 Task3/M2_output.json

📖 Loading data...
   ✅ Loaded 28 trades (Student A)
   ✅ Loaded 33 UPI results (Student B)

   [Test] LEI Check: 5493001KJTIIGC8Y1R12
   Result: True (should be True)

🔍 Performing compliance checks...

📊 Compliance Check Summary
   Total trades: 28

   CFTC status distribution:
      - COMPLIANT: 3
      - NONCOMPLIANT: 22
      - CONDITIONAL: 2
      - NOT_APPLICABLE: 1

   EMIR status distribution:
      - COMPLIANT: 1
      - NONCOMPLIANT: 24
      - NOT_APPLICABLE: 3

   📋 Violation Type Statistics:
      - INVALID_LEI_OTHER: 24 times
      - UPI_NOT_FOUND: 24 times
      - INVALID_LEI_REPORTING: 22 times
      - MISSING_COLLATERAL_CODE: 13 times
      - MISSING_INITIAL_MARGIN: 13 times

In [4]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
Test Script: Custom Trades + Compliance Checks
Design 5 trades, two of which are intentionally invalid.
"""

import json
from pathlib import Path
from datetime import datetime
from typing import Dict, List, Tuple, Any

# ============================================================
# Reuse your compliance check code (copy core functions from original code)
# ============================================================

def lei_checksum(lei: str) -> bool:
    """Validate LEI according to ISO 7064 MOD 97-10 algorithm"""
    if not lei or not isinstance(lei, str):
        return False
    lei = lei.strip().upper()
    if len(lei) != 20:
        return False
    if not lei[:18].isalnum() or not lei[18:].isdigit():
        return False
    
    converted = ""
    for char in lei[:18]:
        if char.isdigit():
            converted += char
        else:
            converted += str(ord(char) - ord('A') + 10)
    
    verification_str = converted + "00"
    try:
        remainder = int(verification_str) % 97
        expected_checksum = 98 - remainder
        expected_checksum_str = f"{expected_checksum:02d}"
        return expected_checksum_str == lei[18:20]
    except (ValueError, ZeroDivisionError):
        return False


def is_valid_lei(lei: Any) -> Tuple[bool, str]:
    """Check if LEI is valid"""
    if lei is None:
        return False, "LEI is missing (null)"
    if not isinstance(lei, str):
        return False, f"LEI must be string, got {type(lei).__name__}"
    lei = lei.strip()
    if lei == "":
        return False, "LEI is empty string"
    if lei == "MISSING_LEI":
        return False, "LEI is explicitly marked as MISSING_LEI"
    if len(lei) != 20:
        return False, f"LEI length is {len(lei)}, expected 20"
    if not lei_checksum(lei):
        return False, "LEI checksum failed (ISO 7064 MOD 97-10)"
    return True, ""


def is_valid_uti(uti: Any) -> Tuple[bool, str]:
    """Check UTI format"""
    if uti is None:
        return False, "UTI is missing (null)"
    if not isinstance(uti, str):
        return False, f"UTI must be string, got {type(uti).__name__}"
    uti = uti.strip()
    if uti == "":
        return False, "UTI is empty string"
    if len(uti) > 52:
        return False, f"UTI length {len(uti)} exceeds 52"
    if len(uti) < 10:
        return False, f"UTI length {len(uti)} is too short"
    return True, ""


def validate_uti_full(uti: str, reporting_lei: str) -> Tuple[bool, str]:
    """Full UTI validation (including namespace check)"""
    if uti is None:
        return False, "UTI is missing"
    if len(uti) < 20:
        return False, f"UTI too short to have namespace: {len(uti)} chars"
    
    namespace = uti[:20]
    if namespace != reporting_lei:
        return False, f"UTI namespace ({namespace}) != reporting LEI ({reporting_lei})"
    
    if len(uti) > 20:
        suffix = uti[20:]
        import re
        if not re.match(r'^[A-Z0-9\-]+$', suffix):
            return False, f"UTI suffix contains invalid chars: {suffix}"
    
    return True, ""


def is_valid_timestamp(ts: Any) -> Tuple[bool, str]:
    if ts is None:
        return False, "Timestamp missing"
    try:
        if ts.endswith('Z'):
            datetime.fromisoformat(ts[:-1] + '+00:00')
        else:
            datetime.fromisoformat(ts)
        return True, ""
    except:
        return False, f"Invalid timestamp: {ts}"


def is_valid_date(date_str: Any) -> Tuple[bool, str]:
    if date_str is None:
        return False, "Date missing"
    if date_str == "9999-99-99":
        return True, ""
    try:
        datetime.strptime(date_str, "%Y-%m-%d")
        return True, ""
    except:
        return False, f"Invalid date: {date_str}"


def is_valid_notional(amount: Any, currency: Any) -> Tuple[bool, str]:
    if amount is None:
        return False, "Notional amount missing"
    if not isinstance(amount, (int, float)) or amount <= 0:
        return False, f"Invalid notional: {amount}"
    if currency == "INVALID_CCY":
        return False, "Invalid currency: INVALID_CCY"
    return True, ""


# ============================================================
# Design 5 custom trades
# ============================================================

def create_custom_trades() -> List[Dict]:
    """
    Design 5 trades, two of which are intentionally invalid:
    
    - CUST001: Fully compliant interest rate swap ✅
    - CUST002: Fully compliant credit default swap ✅
    - CUST003: ❌ Intentional Error 1: Invalid LEI (checksum mismatch)
    - CUST004: ❌ Intentional Error 2: Invalid UTI (namespace does not match reporting LEI)
    - CUST005: Fully compliant commodity forward ✅
    """
    
    valid_lei_1 = "5493001KJTIIGC8Y1R12"   # Valid LEI
    valid_lei_2 = "2138002TXD6KSZ3V5X26"   # Valid LEI (checksum is 26)
    # Note: 2138002TXD6KSZ3V5X26 is a calculated valid LEI
    
    trades = []
    
    # ----------------------------------------------------------
    # CUST001: Fully compliant interest rate swap
    # ----------------------------------------------------------
    trades.append({
        "trade_id": "CUST001",
        "asset_class": "Rates",
        "instrument_type": "Swap",
        "use_case": "Fixed_Float",
        "classification_flag": "CONVENTIONAL_DERIVATIVE",
        "normalized_trade": {
            "trade_id": "CUST001",
            "asset_class": "Rates",
            "instrument_type": "Swap",
            "use_case": "Fixed_Float",
            "reporting_counterparty_lei": valid_lei_1,
            "other_counterparty_lei": valid_lei_2,
            "uti": f"{valid_lei_1}20260514IRS0001",
            "execution_timestamp": "2026-05-14T10:30:00Z",
            "effective_date": "2026-05-16",
            "maturity_date": "2031-05-16",
            "notional_currency": "USD",
            "notional_amount": 25000000,
            "cleared": True,
            "clearing_house": "CME",
            "platform": "SEF",
            "action_type": "NEW",
            "collateral_portfolio_code": "PORT-TEST01",
            "initial_margin_posted": 750000,
            "variation_margin_posted": 0
        }
    })
    
    # ----------------------------------------------------------
    # CUST002: Fully compliant credit default swap
    # ----------------------------------------------------------
    trades.append({
        "trade_id": "CUST002",
        "asset_class": "Credit",
        "instrument_type": "Swap",
        "use_case": "Corporate",
        "classification_flag": "CONVENTIONAL_DERIVATIVE",
        "normalized_trade": {
            "trade_id": "CUST002",
            "asset_class": "Credit",
            "instrument_type": "Swap",
            "use_case": "Corporate",
            "reporting_counterparty_lei": valid_lei_2,
            "other_counterparty_lei": valid_lei_1,
            "uti": f"{valid_lei_2}20260514CDS0001",
            "execution_timestamp": "2026-05-14T09:15:00Z",
            "effective_date": "2026-05-16",
            "maturity_date": "2031-05-16",
            "notional_currency": "EUR",
            "notional_amount": 10000000,
            "cleared": False,
            "platform": "VOICE",
            "action_type": "NEW",
            "collateral_portfolio_code": "PORT-TEST02",
            "initial_margin_posted": 300000,
            "variation_margin_posted": 0
        }
    })
    
    # ----------------------------------------------------------
    # CUST003: ❌ Intentional Error 1 - Invalid LEI (checksum mismatch)
    # The correct LEI should be ...X26, here it's written as ...X27
    # ----------------------------------------------------------
    invalid_lei = "2138002TXD6KSZ3V5X27"  # Checksum should be 26, but here it's 27
    trades.append({
        "trade_id": "CUST003",
        "asset_class": "FX",
        "instrument_type": "Forward",
        "use_case": "NDF",
        "classification_flag": "CONVENTIONAL_DERIVATIVE",
        "normalized_trade": {
            "trade_id": "CUST003",
            "asset_class": "FX",
            "instrument_type": "Forward",
            "use_case": "NDF",
            "reporting_counterparty_lei": valid_lei_1,
            "other_counterparty_lei": invalid_lei,  # ❌ Invalid LEI
            "uti": f"{valid_lei_1}20260514NDF0001",
            "execution_timestamp": "2026-05-14T11:00:00Z",
            "effective_date": "2026-05-16",
            "maturity_date": "2026-11-16",
            "notional_currency": "USD",
            "notional_amount": 5000000,
            "cleared": False,
            "platform": "VOICE",
            "action_type": "NEW",
            "collateral_portfolio_code": "PORT-TEST03",
            "initial_margin_posted": 150000,
            "variation_margin_posted": 0
        }
    })
    
    # ----------------------------------------------------------
    # CUST004: ❌ Intentional Error 2 - Invalid UTI (namespace does not match reporting LEI)
    # The first 20 characters of the UTI should match the reporting LEI, here it's the counterparty LEI
    # ----------------------------------------------------------
    trades.append({
        "trade_id": "CUST004",
        "asset_class": "Equity",
        "instrument_type": "Option",
        "use_case": "SingleName_Call",
        "classification_flag": "CONVENTIONAL_DERIVATIVE",
        "normalized_trade": {
            "trade_id": "CUST004",
            "asset_class": "Equity",
            "instrument_type": "Option",
            "use_case": "SingleName_Call",
            "reporting_counterparty_lei": valid_lei_1,
            "other_counterparty_lei": valid_lei_2,
            "uti": f"{valid_lei_2}20260514OPT0001",  # ❌ Namespace is valid_lei_2, but reporter is valid_lei_1
            "execution_timestamp": "2026-05-14T14:30:00Z",
            "effective_date": "2026-05-16",
            "maturity_date": "2027-05-16",
            "notional_currency": "USD",
            "notional_amount": 2000000,
            "cleared": False,
            "platform": "VOICE",
            "action_type": "NEW",
            "collateral_portfolio_code": "PORT-TEST04",
            "initial_margin_posted": 100000,
            "variation_margin_posted": 0
        }
    })
    
    # ----------------------------------------------------------
    # CUST005: Fully compliant commodity forward
    # ----------------------------------------------------------
    trades.append({
        "trade_id": "CUST005",
        "asset_class": "Commodities",
        "instrument_type": "Forward",
        "use_case": "SingleName",
        "classification_flag": "CONVENTIONAL_DERIVATIVE",
        "normalized_trade": {
            "trade_id": "CUST005",
            "asset_class": "Commodities",
            "instrument_type": "Forward",
            "use_case": "SingleName",
            "reporting_counterparty_lei": valid_lei_2,
            "other_counterparty_lei": valid_lei_1,
            "uti": f"{valid_lei_2}20260514COM0001",
            "execution_timestamp": "2026-05-14T08:00:00Z",
            "effective_date": "2026-05-16",
            "maturity_date": "2027-05-16",
            "notional_currency": "USD",
            "notional_amount": 10000000,
            "underlying_commodity": "WTI_CRUDE",
            "cleared": True,
            "clearing_house": "CME",
            "platform": "SEF",
            "action_type": "NEW",
            "collateral_portfolio_code": "PORT-TEST05",
            "initial_margin_posted": 400000,
            "variation_margin_posted": 0
        }
    })
    
    return trades


# ============================================================
# UPI mock results (for custom trades)
# ============================================================

def create_upi_results_for_custom(trade_ids: List[str]) -> List[Dict]:
    """Create UPI search results for custom trades"""
    upi_results = []
    
    for trade_id in trade_ids:
        if trade_id in ["CUST001", "CUST002", "CUST003", "CUST004", "CUST005"]:
            # Assume all custom trades find a UPI template
            upi_results.append({
                "trade_id": trade_id,
                "status": "FOUND",
                "matched_template": f"Custom.Template.{trade_id}.V1",
                "upi_code": f"UPI{trade_id[-3:]}ABCDEFGHIJ",  # Mock a 20-character UPI code
                "classification_note": None
            })
    
    return upi_results


# ============================================================
# Compliance check functions
# ============================================================

def check_cftc_compliance(trade: Dict, upi_result: Dict) -> Tuple[str, List[str]]:
    """CFTC compliance check"""
    violations = []
    
    # UTI check (full version)
    uti = trade.get("uti")
    reporting_lei = trade.get("reporting_counterparty_lei")
    if not uti:
        violations.append("MISSING_UTI")
    else:
        uti_valid, uti_error = validate_uti_full(uti, reporting_lei)
        if not uti_valid:
            violations.append(f"INVALID_UTI: {uti_error}")
    
    # LEI check
    lei_valid, lei_error = is_valid_lei(trade.get("reporting_counterparty_lei"))
    if not lei_valid:
        violations.append(f"INVALID_REPORTING_LEI: {lei_error}")
    
    other_lei = trade.get("other_counterparty_lei")
    if other_lei:
        lei_other_valid, lei_other_error = is_valid_lei(other_lei)
        if not lei_other_valid:
            violations.append(f"INVALID_OTHER_LEI: {lei_other_error}")
    
    # Timestamp
    ts_valid, ts_error = is_valid_timestamp(trade.get("execution_timestamp"))
    if not ts_valid:
        violations.append(f"INVALID_TIMESTAMP: {ts_error}")
    
    # Dates
    effective_valid, effective_error = is_valid_date(trade.get("effective_date"))
    if not effective_valid:
        violations.append(f"INVALID_EFFECTIVE_DATE: {effective_error}")
    
    maturity_valid, maturity_error = is_valid_date(trade.get("maturity_date"))
    if not maturity_valid:
        violations.append(f"INVALID_MATURITY_DATE: {maturity_error}")
    
    # Notional amount
    notional_valid, notional_error = is_valid_notional(
        trade.get("notional_amount"),
        trade.get("notional_currency")
    )
    if not notional_valid:
        violations.append(f"INVALID_NOTIONAL: {notional_error}")
    
    # Cleared field
    cleared = trade.get("cleared")
    if cleared not in [True, False]:
        violations.append(f"INVALID_CLEARED: {cleared}")
    
    # UPI status
    upi_status = upi_result.get("status")
    if upi_status != "FOUND":
        violations.append(f"UPI_{upi_status}")
    
    status = "COMPLIANT" if len(violations) == 0 else "NONCOMPLIANT"
    return status, violations


def check_emir_compliance(trade: Dict, upi_result: Dict) -> Tuple[str, List[str]]:
    """EMIR compliance check"""
    violations = []
    
    # All CFTC checks
    cftc_status, cftc_violations = check_cftc_compliance(trade, upi_result)
    violations.extend(cftc_violations)
    
    # EMIR additional fields: collateral code
    if not trade.get("collateral_portfolio_code"):
        violations.append("MISSING_COLLATERAL_CODE")
    
    # Initial margin
    init_margin = trade.get("initial_margin_posted")
    if init_margin is None:
        violations.append("MISSING_INITIAL_MARGIN")
    elif not isinstance(init_margin, (int, float)) or init_margin < 0:
        violations.append(f"INVALID_INITIAL_MARGIN: {init_margin}")
    
    # Variation margin
    var_margin = trade.get("variation_margin_posted")
    if var_margin is None:
        violations.append("MISSING_VARIATION_MARGIN")
    elif not isinstance(var_margin, (int, float)):
        violations.append(f"INVALID_VARIATION_MARGIN: {var_margin}")
    
    status = "COMPLIANT" if len(violations) == 0 else "NONCOMPLIANT"
    return status, violations


# ============================================================
# Main function
# ============================================================

def main():
    print("=" * 60)
    print("Custom Trades Compliance Check Test")
    print("=" * 60)
    
    # 1. Create custom trades
    print("\n📝 Creating custom trades...")
    custom_trades = create_custom_trades()
    print(f"   ✅ {len(custom_trades)} trades created")
    
    # 2. Display trade list
    print("\n📋 Trade List:")
    for trade in custom_trades:
        norm = trade["normalized_trade"]
        print(f"   - {norm['trade_id']}: {norm['asset_class']}/{norm['instrument_type']}")
    
    # 3. Create UPI results
    trade_ids = [t["normalized_trade"]["trade_id"] for t in custom_trades]
    upi_results = create_upi_results_for_custom(trade_ids)
    upi_index = {r["trade_id"]: r for r in upi_results}
    
    # 4. Perform compliance checks
    print("\n🔍 Performing compliance checks...")
    
    results = {
        "summary": {
            "total_trades": len(custom_trades),
            "cftc_compliant": 0,
            "cftc_noncompliant": 0,
            "emir_compliant": 0,
            "emir_noncompliant": 0,
            "violation_type_counts": {}
        },
        "details": [],
        "design_notes": {
            "intentionally_invalid_trades": ["CUST003", "CUST004"],
            "error_types": {
                "CUST003": "Invalid LEI (checksum mismatch: expected 26, got 27)",
                "CUST004": "Invalid UTI (namespace does not match reporting counterparty LEI)"
            }
        }
    }
    
    for trade_wrapper in custom_trades:
        trade = trade_wrapper["normalized_trade"]
        trade_id = trade["trade_id"]
        upi_result = upi_index.get(trade_id, {"status": "NOT_FOUND"})
        
        cftc_status, cftc_violations = check_cftc_compliance(trade, upi_result)
        emir_status, emir_violations = check_emir_compliance(trade, upi_result)
        
        # Statistics
        if cftc_status == "COMPLIANT":
            results["summary"]["cftc_compliant"] += 1
        else:
            results["summary"]["cftc_noncompliant"] += 1
        
        if emir_status == "COMPLIANT":
            results["summary"]["emir_compliant"] += 1
        else:
            results["summary"]["emir_noncompliant"] += 1
        
        for v in cftc_violations + emir_violations:
            v_code = v.split(":")[0] if ":" in v else v
            results["summary"]["violation_type_counts"][v_code] = \
                results["summary"]["violation_type_counts"].get(v_code, 0) + 1
        
        results["details"].append({
            "trade_id": trade_id,
            "cftc": {
                "status": cftc_status,
                "violations": cftc_violations
            },
            "emir": {
                "status": emir_status,
                "violations": emir_violations
            },
            "trade_data": {
                "asset_class": trade.get("asset_class"),
                "instrument_type": trade.get("instrument_type"),
                "reporting_lei": trade.get("reporting_counterparty_lei"),
                "other_lei": trade.get("other_counterparty_lei"),
                "uti": trade.get("uti")
            }
        })
    
    # 5. Output results
    print("\n" + "=" * 60)
    print("📊 Check Results Summary")
    print("=" * 60)
    print(f"   Total number of trades: {results['summary']['total_trades']}")
    print(f"   CFTC Compliant: {results['summary']['cftc_compliant']}")
    print(f"   CFTC Noncompliant: {results['summary']['cftc_noncompliant']}")
    print(f"   EMIR Compliant: {results['summary']['emir_compliant']}")
    print(f"   EMIR Noncompliant: {results['summary']['emir_noncompliant']}")
    
    print("\n   🎯 Design Notes:")
    print("      - Intentionally invalid trades: CUST003, CUST004")
    print("      - CUST003: LEI checksum mismatch (expected 26, got 27)")
    print("      - CUST004: UTI namespace does not match reporting LEI")
    
    if results['summary']['violation_type_counts']:
        print("\n   📋 Violation Type Statistics:")
        for v_code, count in sorted(results['summary']['violation_type_counts'].items(), key=lambda x: -x[1]):
            print(f"      - {v_code}: {count} occurrences")
    
    # 6. Detailed report
    print("\n📝 Detailed Results:")
    for detail in results["details"]:
        print(f"\n   [{detail['trade_id']}]")
        print(f"      CFTC: {detail['cftc']['status']}")
        if detail['cftc']['violations']:
            for v in detail['cftc']['violations']:
                print(f"         - {v}")
        print(f"      EMIR: {detail['emir']['status']}")
        if detail['emir']['violations']:
            for v in detail['emir']['violations']:
                print(f"         - {v}")
    
    # 7. Save results
    output_file = Path("/Users/sunyifan/Desktop/6822 Task3/custom_trades_report.json")
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(results, f, indent=2, ensure_ascii=False)
    
    print(f"\n💾 Detailed report saved to: {output_file}")
    print("\n" + "=" * 60)
    print("✅ Test complete!")
    print("=" * 60)


if __name__ == "__main__":
    main()

Custom Trades Compliance Check Test

📝 Creating custom trades...
   ✅ 5 trades created

📋 Trade List:
   - CUST001: Rates/Swap
   - CUST002: Credit/Swap
   - CUST003: FX/Forward
   - CUST004: Equity/Option
   - CUST005: Commodities/Forward

🔍 Performing compliance checks...

📊 Check Results Summary
   Total number of trades: 5
   CFTC Compliant: 3
   CFTC Noncompliant: 2
   EMIR Compliant: 3
   EMIR Noncompliant: 2

   🎯 Design Notes:
      - Intentionally invalid trades: CUST003, CUST004
      - CUST003: LEI checksum mismatch (expected 26, got 27)
      - CUST004: UTI namespace does not match reporting LEI

   📋 Violation Type Statistics:
      - INVALID_OTHER_LEI: 2 occurrences
      - INVALID_UTI: 2 occurrences

📝 Detailed Results:

   [CUST001]
      CFTC: COMPLIANT
      EMIR: COMPLIANT

   [CUST002]
      CFTC: COMPLIANT
      EMIR: COMPLIANT

   [CUST003]
      CFTC: NONCOMPLIANT
         - INVALID_OTHER_LEI: LEI checksum failed (ISO 7064 MOD 97-10)
      EMIR: NONCOMPLIANT
    